In [2]:
import sys
import os
sys.path.append(os.path.abspath('..'))

#Importando os arquivos de pre-processamento e download de dados
from pre_processamento import *
from data_downloading import *
from collections import Counter
from vector_model import *
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import precision_score, recall_score, f1_score
from recuperar import *


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/agnesmva/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
#Download dos dados
colecao_subset_removida = download_data_subset_removida()

Carregando colecao_subset_removida.pkl do cache local...


In [4]:
#O processo de lematização é aplicado a cada documento da coleção, resultando em uma nova lista de documentos lematizados.
#Atualmente é o processo mais demorado de pre-processamento. 
#Resolvi dividir o processo de pre-processamento em etapas para facilitar a identificação de gargalos e otimizar o código posteriormente, se necessário.
lema_subset = [preprocess_lemma(doc) for doc in colecao_subset_removida.data]

In [5]:
steaming_subset = [preprocess_stemming(doc) for doc in colecao_subset_removida.data]

In [6]:
original_subset = [preprocess_original(doc) for doc in colecao_subset_removida.data]

In [7]:
print("Original:")
print(sum(len(doc) for doc in original_subset))

print("Steamming:")
print(sum(len(doc) for doc in steaming_subset))

print("Lema:")
print(sum(len(doc) for doc in lema_subset))

Original:
2102451
Steamming:
1088804
Lema:
902267


Vamos verificar os termos únicos após o processo de pré-processamento

In [8]:
vocab_original = set(token for doc in original_subset for token in doc)
vocab_stemming = set(token for doc in steaming_subset for token in doc)
vocab_lemma = set(token for doc in lema_subset for token in doc)

print(f"Original: {len(vocab_original):,}")
print(f"Stemming: {len(vocab_stemming):,}")
print(f"Lematização: {len(vocab_lemma):,}")

Original: 190,592
Stemming: 55,170
Lematização: 49,458


In [9]:
orig = len(vocab_original)
stem = len(vocab_stemming)
lemma = len(vocab_lemma)

print(f"Redução Stemming: {(orig-stem)/orig*100:.2f}%")
print(f"Redução Lematização: {(orig-lemma)/orig*100:.2f}%")

Redução Stemming: 71.05%
Redução Lematização: 74.05%


Enquanto executávamos o pre-processamento resolvemos verificar como ficaram as 20 primeiras palavras que mais apareciam. Foi bom, pois conseguimos identificar problemas na nossa função de stemming. O curioso é que ficou como termo mais retornando a letra X, não sabemos o porquê.

In [10]:
freq_original = Counter(
    token
    for doc in original_subset
    for token in doc
)

freq_stemming = Counter(
    token
    for doc in steaming_subset
    for token in doc
)

freq_lemma = Counter(
    token
    for doc in lema_subset
    for token in doc
)

print(freq_original.most_common(20))
print(freq_stemming.most_common(20))
print(freq_lemma.most_common(20))

[('the', 104925), ('to', 52140), ('of', 46496), ('a', 42132), ('and', 41381), ('in', 30043), ('is', 29275), ('i', 27603), ('that', 25833), ('for', 19436), ('it', 17517), ('you', 15710), ('on', 13686), ('be', 13459), ('this', 13041), ('are', 12629), ('have', 12605), ('with', 12314), ('not', 11480), ('as', 11027)]
[('use', 7212), ('one', 6781), ('would', 6165), ('max', 4680), ('get', 4523), ('like', 4498), ('peopl', 4137), ('know', 3916), ('time', 3757), ('think', 3400), ('also', 3352), ('say', 3056), ('make', 3040), ('work', 2928), ('year', 2854), ('system', 2676), ('file', 2635), ('new', 2626), ('good', 2552), ('could', 2537)]
[('know', 4590), ('people', 4132), ('like', 4049), ('think', 3860), ('use', 3594), ('time', 3522), ('good', 3340), ('say', 2916), ('work', 2842), ('year', 2817), ('new', 2680), ('go', 2595), ('file', 2566), ('system', 2470), ('want', 2423), ('come', 2406), ('look', 2348), ('need', 2317), ('find', 2298), ('problem', 2297)]


Agora, após o pré-processamento do corpus. 

In [11]:
vocabulary, term_to_idx = build_vocabulary(lema_subset)
print(f"Vocabulário: {len(vocabulary)} termos")
print(vocabulary[:5])
print(term_to_idx)

Vocabulário: 49458 termos
['aaa', 'aaaaaaaaaaaa', 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaauuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuugggggggggggggggg', 'aaah', 'aaahh']
{'aaa': 0, 'aaaaaaaaaaaa': 1, 'aaaaaaaaaaaaaaaaaaaaaaaaaaaaaaauuuuuuuuuuuuuuuuuuuuuuuuuuuuuuuugggggggggggggggg': 2, 'aaah': 3, 'aaahh': 4, 'aaahhhh': 5, 'aaai': 6, 'aacc': 7, 'aah': 8, 'aalternate': 9, 'aamazing': 10, 'aammmaaaazzzzzziinnnnggggg': 11, 'aan': 12, 'aanbieden': 13, 'aanerud': 14, 'aangeboden': 15, 'aangegeven': 16, 'aangezien': 17, 'aantal': 18, 'aao': 19, 'aap': 20, 'aargh': 21, 'aaron': 22, 'aaronson': 23, 'aarp': 24, 'aarseth': 25, 'aarskog': 26, 'aas': 27, 'aaske': 28, 'aavso': 29, 'ab': 30, 'ababs': 31, 'abandon': 32, 'abandond': 33, 'abate': 34, 'abatement': 35, 'abba': 36, 'abbasids': 37, 'abberant': 38, 'abberation': 39, 'abbey': 40, 'abbie': 41, 'abbot': 42, 'abbott': 43, 'abbreviate': 44, 'abbreviation': 45, 'abbreviations': 46, 'abc': 47, 'abcdef': 48, 'abd': 49, 'abdel': 50, 'abdi': 51, 'abdication': 52, 'abdo': 

In [12]:
# --- Passo 1: Calcular DF ---
df_vector = calcular_frequencia_documento(lema_subset, vocabulary, term_to_idx)
print(f"\nDF para os 5 primeiros termos: {df_vector[:5]}")

# --- Passo 2: Calcular IDF ---
idf_vector = calcular_idf(df_vector, total_documentos=len(lema_subset))
print(f"\nIDF para os 5 primeiros termos: {idf_vector[:5]}")

# --- Passo 3: Construir as Matrizes Finais ---
tf_matrix, tfidf_matrix = calcular_tf_idf_matriz(lema_subset, vocabulary, term_to_idx, idf_vector)
print(f"\nFormato da matriz TF-IDF: {len(tfidf_matrix)} documentos x {len(vocabulary)} colunas.")


DF para os 5 primeiros termos: [25, 1, 1, 1, 1]

IDF para os 5 primeiros termos: [7.075788020046156, 9.640737377507692, 9.640737377507692, 9.640737377507692, 9.640737377507692]

Formato da matriz TF-IDF: 11314 documentos x 49458 colunas.


Pra testar a função de calcular corretamente o TF-IDF da query coloquei esse exemplo. 
ela retorna em todo o documento, e no inicio do documento estão indexados termos como aaa aaa, etc. que não servem pra nossa busca, por isso coloquei o if para retornar o tf-idf no termo médio e retorna esses valores.

In [13]:
query = "There's so many things that you could be, but are you in it for the love or are you just in it for the money?"
query_tokens = preprocess_lemma(query)
query_vector = calcular_tf_idf_query(query_tokens, vocabulary, term_to_idx, idf_vector)
for query_term, value in zip(vocabulary, query_vector):
    if value > 0:
        print(f"Termo: '{query_term}', TF-IDF: {value:.4f}")

Termo: 'love', TF-IDF: 4.5750
Termo: 'money', TF-IDF: 4.3911
Termo: 'thing', TF-IDF: 3.0388


In [14]:
query_vector = normalize(query_vector)

ranking = rank_documents(
    query_vector,
    tfidf_matrix,
)

print(ranking[:10])

[(6458, 0.39252857979220057), (7978, 0.3812357773788567), (4884, 0.3667032175225678), (10406, 0.2997516218408424), (5488, 0.26498097626970235), (10965, 0.25990221543765246), (592, 0.24502703400912842), (2029, 0.23189410417757292), (8179, 0.22886841626532545), (9447, 0.22501168885710301)]


In [15]:
resultados_alinhados = buscar_documentos(query_vector, tfidf_matrix, 3)
print(f"Resultados para a busca: '{query}'\n" + "="*50)

for posicao, (idx, score) in enumerate(resultados_alinhados, 1):
    print(f"\n{posicao}º Lugar - Documento Índice [{idx}] | Score de Similaridade: {score:.4f}")
    print("-"*50)
    
    # Buscamos o texto original usando o índice 'idx' retornado pelo buscador
    texto_original = colecao_subset_removida.data[idx]
    
    # Printa apenas os primeiros 300 caracteres para não inundar a tela
    print(texto_original[:300] + "...")

Resultados para a busca: 'There's so many things that you could be, but are you in it for the love or are you just in it for the money?'

1º Lugar - Documento Índice [6458] | Score de Similaridade: 0.3925
--------------------------------------------------
So long as we think that good things are what we *have* to do rather than
what we come to *want* to do, we miss the point. The more we love God; the
more we come to love what and whom He loves.

When I find that what I am doing is not good, it is not a sign to try
even harder (Romans 7:14-8:2); it i...

2º Lugar - Documento Índice [7978] | Score de Similaridade: 0.3812
--------------------------------------------------
Above all, love each other deeply, because love covers over a multitude of
sins. ...

3º Lugar - Documento Índice [4884] | Score de Similaridade: 0.3667
--------------------------------------------------
davem@bnr.ca (Dave Mielke) writes,

 
  
I am extremely uncomfortable with this way of phrasing it.  God's love 
is u